# Chart Storytelling

Notebook simplificado para cargar, preparar y unificar los datos de producción, ventas y exportaciones de vehículos.

**Flujo de trabajo:** carga de catálogos → consolidación de archivos → preparación → unión de registros coincidentes → limpieza.

## 1. Librerías y catálogos

In [1]:
import glob
import re
import pandas as pd

# Catálogos utilizados para traducir identificadores a descripciones
tc_mes = pd.read_csv("data/hibrid/catalogo/tc_mes.csv")
tc_entidad_federativa = pd.read_csv("data/hibrid/catalogo/tc_entidad_federativa.csv")
tc_pais_origen = pd.read_csv("data/venta/catalogo/tc_pais_origen.csv")
tc_pais_destino = pd.read_csv("data/export/catalogo/tc_pais_destino.csv")

## 2. Carga y consolidación de los datasets

In [2]:
def cargar_archivos(pattern, catalogos=None, renombres=None):
    """Lee archivos anuales, los concatena y agrega descripciones de catálogos."""
    archivos = sorted(glob.glob(pattern))
    if not archivos:
        raise FileNotFoundError(f"No se encontraron archivos para: {pattern}")

    dataframes = []
    for archivo in archivos:
        df = pd.read_csv(archivo)
        year = re.search(r"\d{4}", archivo)
        df["YEAR_SOURCE"] = int(year.group()) if year else None
        dataframes.append(df)

    resultado = pd.concat(dataframes, ignore_index=True)

    # Incorporar descripciones cuando el identificador correspondiente exista
    for columna_id, catalogo, columna_nombre in (catalogos or []):
        if columna_id in resultado.columns:
            resultado = resultado.merge(catalogo, on=columna_id, how="left")
            resultado.drop(columns=columna_id, inplace=True)
            if columna_nombre:
                resultado.rename(columns=columna_nombre, inplace=True)

    if renombres:
        resultado.rename(columns=renombres, inplace=True)

    print(f"{len(archivos)} archivos cargados | {len(resultado):,} registros")
    return resultado


# Producción
df_production_all = cargar_archivos(
    "data/prod/data/raiavl_produccion_mensual_tr_cifra_*.csv",
    catalogos=[
        ("ID_MES", tc_mes, {"DESCRIPCION_MES": "MES"}),
        ("ID_ENTIDAD", tc_entidad_federativa, {"DESC_ENTIDAD": "ENTIDAD_FEDERATIVA"})
    ]
)

# Ventas
df_sales_all = cargar_archivos(
    "data/venta/data/raiavl_venta_mensual_tr_cifra_*.csv",
    catalogos=[
        ("ID_MES", tc_mes, {"DESCRIPCION_MES": "MES"}),
        ("ID_PAIS_ORIGEN", tc_pais_origen, {
            "DESCRIPCION_PAIS": "PAIS_ORIGEN",
            "DESC_PAIS": "PAIS_ORIGEN"
        })
    ]
)

# Exportaciones
df_export_all = cargar_archivos(
    "data/export/data/raiavl_exportacion_mensual_tr_cifra_*.csv",
    catalogos=[
        ("ID_MES", tc_mes, {"DESCRIPCION_MES": "MES"}),
        ("ID_PAIS_DESTINO", tc_pais_destino, {
            "DESCRIPCION_PAIS": "PAIS_DESTINO",
            "DESC_PAIS_DESTINO": "PAIS_DESTINO",
            "DESC_PAIS": "PAIS_DESTINO"
        })
    ]
)

22 archivos cargados | 18,063 registros
22 archivos cargados | 108,195 registros
22 archivos cargados | 194,088 registros


## 3. Inspección y preparación de datos

In [3]:
# Revisar dimensiones y columnas principales
for nombre, df in {
    "Producción": df_production_all,
    "Ventas": df_sales_all,
    "Exportaciones": df_export_all
}.items():
    print(f"\n{nombre}: {df.shape[0]:,} filas y {df.shape[1]} columnas")
    print(list(df.columns))
    display(df.head(3))


Producción: 18,063 filas y 11 columnas
['PROD_EST', 'COBERTURA', 'ANIO', 'MARCA', 'MODELO', 'TIPO', 'SEGMENTO', 'UNI_VEH', 'ESTATUS', 'YEAR_SOURCE', 'MES']


,PROD_EST,COBERTURA,ANIO,MARCA,MODELO,TIPO,SEGMENTO,UNI_VEH,ESTATUS,YEAR_SOURCE,MES
0,Registro administrativo de la industria automo...,Nacional,2005,Chrysler,PT Cruiser,Automóviles,Compactos,7397,Cifras Definitivas,2005,Enero
1,Registro administrativo de la industria automo...,Nacional,2005,Chrysler,RAM 1500-,Camiones ligeros,Pick Ups,536,Cifras Definitivas,2005,Enero
2,Registro administrativo de la industria automo...,Nacional,2005,Chrysler,RAM 2500-,Camiones ligeros,Pick Ups,6124,Cifras Definitivas,2005,Enero



Ventas: 108,195 filas y 13 columnas
['PROD_EST', 'COBERTURA', 'ANIO', 'MARCA', 'MODELO', 'TIPO', 'SEGMENTO', 'ORIGEN', 'UNI_VEH', 'ESTATUS', 'YEAR_SOURCE', 'MES', 'DESC_PAIS_ORIGEN']


,PROD_EST,COBERTURA,ANIO,MARCA,MODELO,TIPO,SEGMENTO,ORIGEN,UNI_VEH,ESTATUS,YEAR_SOURCE,MES,DESC_PAIS_ORIGEN
0,Registro Administrativo de la Industria Automo...,Nacional,2005,Acura,Mdx,Camiones ligeros,SUV's,IMPORTADO,18,Cifras Definitivas,2005,Enero,Canadá
1,Registro Administrativo de la Industria Automo...,Nacional,2005,Acura,Rl,Automóviles,De Lujo,IMPORTADO,4,Cifras Definitivas,2005,Enero,Japón
2,Registro Administrativo de la Industria Automo...,Nacional,2005,Acura,Tl 3.5l,Automóviles,De Lujo,IMPORTADO,38,Cifras Definitivas,2005,Enero,Estados Unidos



Exportaciones: 194,088 filas y 12 columnas
['PROD_EST', 'COBERTURA', 'ANIO', 'MARCA', 'MODELO', 'TIPO', 'SEGMENTO', 'UNI_VEH', 'ESTATUS', 'YEAR_SOURCE', 'MES', 'PAIS_DESTINO']


,PROD_EST,COBERTURA,ANIO,MARCA,MODELO,TIPO,SEGMENTO,UNI_VEH,ESTATUS,YEAR_SOURCE,MES,PAIS_DESTINO
0,Registro Administrativo de la Industria Automo...,Nacional,2005,Chrysler,PT Cruiser,Automóviles,Compactos,2186,Cifras Definitivas,2005,Enero,Alemania
1,Registro Administrativo de la Industria Automo...,Nacional,2005,Chrysler,PT Cruiser,Automóviles,Compactos,86,Cifras Definitivas,2005,Enero,Canadá
2,Registro Administrativo de la Industria Automo...,Nacional,2005,Chrysler,PT Cruiser,Automóviles,Compactos,4904,Cifras Definitivas,2005,Enero,Estados Unidos


## 4. Unificación de los datasets

Se utilizan las variables compartidas para relacionar los tres conjuntos de datos. El `inner merge` conserva únicamente las combinaciones que aparecen en producción, ventas y exportaciones; se evita el `outer merge` completo que generaba un volumen elevado de registros.

In [4]:
merge_keys = ["ANIO", "MES", "MARCA", "MODELO", "TIPO", "SEGMENTO"]

# Verificar que todas las claves estén disponibles
for nombre, df in {
    "Producción": df_production_all,
    "Ventas": df_sales_all,
    "Exportaciones": df_export_all
}.items():
    faltantes = [col for col in merge_keys if col not in df.columns]
    if faltantes:
        raise KeyError(f"Faltan claves en {nombre}: {faltantes}")

# Renombrar la métrica para distinguir su origen
df_prod_temp = df_production_all.rename(columns={"UNI_VEH": "UNI_VEH_PRODUCCION"})
df_sales_temp = df_sales_all.rename(columns={"UNI_VEH": "UNI_VEH_VENTAS"})
df_export_temp = df_export_all.rename(columns={"UNI_VEH": "UNI_VEH_EXPORTACION"})

# Unir únicamente registros coincidentes en los tres datasets
df_merged_inner = (
    df_prod_temp
    .merge(df_sales_temp, on=merge_keys, how="inner", suffixes=("_PROD", "_VENTA"))
    .merge(df_export_temp, on=merge_keys, how="inner", suffixes=("", "_EXP"))
)

print(f"Registros después de la unión: {len(df_merged_inner):,}")
display(df_merged_inner.head())

Registros después de la unión: 166,458


,PROD_EST_PROD,COBERTURA_PROD,ANIO,MARCA,MODELO,TIPO,SEGMENTO,UNI_VEH_PRODUCCION,ESTATUS_PROD,YEAR_SOURCE_PROD,...,UNI_VEH_VENTAS,ESTATUS_VENTA,YEAR_SOURCE_VENTA,DESC_PAIS_ORIGEN,PROD_EST,COBERTURA,UNI_VEH_EXPORTACION,ESTATUS,YEAR_SOURCE,PAIS_DESTINO
0,Registro administrativo de la industria automo...,Nacional,2005,Chrysler,PT Cruiser,Automóviles,Compactos,7397,Cifras Definitivas,2005,...,148,Cifras Definitivas,2005,México,Registro Administrativo de la Industria Automo...,Nacional,2186,Cifras Definitivas,2005,Alemania
1,Registro administrativo de la industria automo...,Nacional,2005,Chrysler,PT Cruiser,Automóviles,Compactos,7397,Cifras Definitivas,2005,...,148,Cifras Definitivas,2005,México,Registro Administrativo de la Industria Automo...,Nacional,86,Cifras Definitivas,2005,Canadá
2,Registro administrativo de la industria automo...,Nacional,2005,Chrysler,PT Cruiser,Automóviles,Compactos,7397,Cifras Definitivas,2005,...,148,Cifras Definitivas,2005,México,Registro Administrativo de la Industria Automo...,Nacional,4904,Cifras Definitivas,2005,Estados Unidos
3,Registro administrativo de la industria automo...,Nacional,2005,Chrysler,RAM 4000,Camiones ligeros,Pick Ups,4729,Cifras Definitivas,2005,...,321,Cifras Definitivas,2005,Estados Unidos,Registro Administrativo de la Industria Automo...,Nacional,539,Cifras Definitivas,2005,Canadá
4,Registro administrativo de la industria automo...,Nacional,2005,Chrysler,RAM 4000,Camiones ligeros,Pick Ups,4729,Cifras Definitivas,2005,...,321,Cifras Definitivas,2005,Estados Unidos,Registro Administrativo de la Industria Automo...,Nacional,4059,Cifras Definitivas,2005,Estados Unidos


## 5. Limpieza del dataset unificado

In [5]:
df_merged_clean = df_merged_inner.copy()

# Unificar columnas repetidas provenientes de los distintos datasets
suffixes = ["_PROD", "_VENTA", "_EXP"]

for base in ["PROD_EST", "COBERTURA", "ESTATUS", "YEAR_SOURCE"]:
    columnas = [
        col for col in df_merged_clean.columns
        if col == base or col in [f"{base}{s}" for s in suffixes]
    ]

    if columnas:
        df_merged_clean[base] = df_merged_clean[columnas].bfill(axis=1).iloc[:, 0]
        redundantes = [col for col in columnas if col != base]
        df_merged_clean.drop(columns=redundantes, inplace=True)

print(f"Dataset final: {len(df_merged_clean):,} filas y {len(df_merged_clean.columns)} columnas")
print("Columnas finales:", list(df_merged_clean.columns))
display(df_merged_clean.head(10))

Dataset final: 166,458 filas y 16 columnas
Columnas finales: ['ANIO', 'MARCA', 'MODELO', 'TIPO', 'SEGMENTO', 'UNI_VEH_PRODUCCION', 'MES', 'ORIGEN', 'UNI_VEH_VENTAS', 'DESC_PAIS_ORIGEN', 'PROD_EST', 'COBERTURA', 'UNI_VEH_EXPORTACION', 'ESTATUS', 'YEAR_SOURCE', 'PAIS_DESTINO']


,ANIO,MARCA,MODELO,TIPO,SEGMENTO,UNI_VEH_PRODUCCION,MES,ORIGEN,UNI_VEH_VENTAS,DESC_PAIS_ORIGEN,PROD_EST,COBERTURA,UNI_VEH_EXPORTACION,ESTATUS,YEAR_SOURCE,PAIS_DESTINO
0,2005,Chrysler,PT Cruiser,Automóviles,Compactos,7397,Enero,NACIONAL,148,México,Registro administrativo de la industria automo...,Nacional,2186,Cifras Definitivas,2005,Alemania
1,2005,Chrysler,PT Cruiser,Automóviles,Compactos,7397,Enero,NACIONAL,148,México,Registro administrativo de la industria automo...,Nacional,86,Cifras Definitivas,2005,Canadá
2,2005,Chrysler,PT Cruiser,Automóviles,Compactos,7397,Enero,NACIONAL,148,México,Registro administrativo de la industria automo...,Nacional,4904,Cifras Definitivas,2005,Estados Unidos
3,2005,Chrysler,RAM 4000,Camiones ligeros,Pick Ups,4729,Enero,IMPORTADO,321,Estados Unidos,Registro administrativo de la industria automo...,Nacional,539,Cifras Definitivas,2005,Canadá
4,2005,Chrysler,RAM 4000,Camiones ligeros,Pick Ups,4729,Enero,IMPORTADO,321,Estados Unidos,Registro administrativo de la industria automo...,Nacional,4059,Cifras Definitivas,2005,Estados Unidos
5,2005,Ford Motor,Focus ZX3,Automóviles,Compactos,2563,Enero,NACIONAL,1,México,Registro administrativo de la industria automo...,Nacional,2380,Cifras Definitivas,2005,Estados Unidos
6,2005,General Motors,Avalanche UUV,Camiones ligeros,Pick Ups,3676,Enero,NACIONAL,45,México,Registro administrativo de la industria automo...,Nacional,18,Cifras Definitivas,2005,Argentina
7,2005,General Motors,Avalanche UUV,Camiones ligeros,Pick Ups,3676,Enero,NACIONAL,45,México,Registro administrativo de la industria automo...,Nacional,1,Cifras Definitivas,2005,Aruba
8,2005,General Motors,Avalanche UUV,Camiones ligeros,Pick Ups,3676,Enero,NACIONAL,45,México,Registro administrativo de la industria automo...,Nacional,0,Cifras Definitivas,2005,Bahamas
9,2005,General Motors,Avalanche UUV,Camiones ligeros,Pick Ups,3676,Enero,NACIONAL,45,México,Registro administrativo de la industria automo...,Nacional,416,Cifras Definitivas,2005,Canadá


In [6]:
csv_path = "dataset_final_raiavl.csv"
df_merged_clean.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Saved {len(df_merged_clean):,} rows to {csv_path}")

Saved 166,458 rows to dataset_final_raiavl.csv


In [1]:
import numpy as np
import pandas as pd

from bokeh.plotting import figure, show
from bokeh.io import output_notebook, output_file, save
from bokeh.models import (
    ColumnDataSource, Label, Span, BoxAnnotation, Title,
    HoverTool, NumeralTickFormatter, Range1d
)
from bokeh.layouts import column
from bokeh.palettes import Category10

output_notebook()

# -------------------------------------------------------------------------
# 1. Paleta Storytelling y funciones auxiliares (estilo Dr. Sabur Butt)
# -------------------------------------------------------------------------
GREY       = "#A6A6A6"   # datos de contexto
LIGHT_GREY = "#D9D9D9"   # ejes y series secundarias
TEXT_GREY  = "#7F7F7F"   # etiquetas de ejes y notas
DARK       = "#404040"   # texto principal y títulos
BLUE       = "#1F5A96"   # protagonista positivo (El Gran Pivote SUV)
LIGHT_BLUE = "#9DC3E6"
ORANGE     = "#E46C0A"   # alerta estratégica (Choque arancelario 2026)

def declutter(p, x_axis=True, y_axis=True):
    """Estilo explicativo: sin toolbar, sin grid, sin bordes, ejes suaves."""
    p.toolbar_location = None
    p.outline_line_color = None
    p.grid.grid_line_color = None
    p.axis.minor_tick_line_color = None
    p.axis.major_tick_line_color = None
    p.axis.axis_line_color = LIGHT_GREY
    p.axis.major_label_text_color = TEXT_GREY
    p.axis.axis_label_text_color = TEXT_GREY
    p.axis.axis_label_text_font_style = "normal"
    p.xaxis.visible = x_axis
    p.yaxis.visible = y_axis
    return p

def action_title(p, headline, subtitle=None):
    """Título de acción ('so what') + subtítulo explicativo."""
    p.title.text = ""
    if subtitle:
        p.add_layout(Title(
            text=subtitle, text_font_size="12px",
            text_color=TEXT_GREY, text_font_style="normal"
        ), "above")
    p.add_layout(Title(
        text=headline, text_font_size="15px",
        text_color=DARK, text_font_style="bold"
    ), "above")
    return p

# -------------------------------------------------------------------------
# 2. Carga y deduplicación exacta de dataset_final_raiavl.csv
# -------------------------------------------------------------------------
df = pd.read_csv("dataset_final_raiavl.csv")
keys_base = ["ANIO", "MES", "MARCA", "MODELO", "TIPO", "SEGMENTO"]

# Deduplicar producción antes de agregar (evita multiplicación por PAIS_DESTINO)
df_prod = df.drop_duplicates(subset=keys_base)[
    keys_base + ["UNI_VEH_PRODUCCION", "ESTATUS"]
].copy()

piv_all = df_prod.pivot_table(
    index="ANIO", columns="SEGMENTO", values="UNI_VEH_PRODUCCION",
    aggfunc="sum", fill_value=0
).reset_index()

piv_25 = piv_all[piv_all["ANIO"] <= 2025].copy()

# Comparación justa Ene-Oct (YTD) para 2025 vs 2026 (evita sesgo estacional)
meses_2026 = df_prod[df_prod["ANIO"] == 2026]["MES"].unique()
piv_ytd = df_prod[
    df_prod["MES"].isin(meses_2026) & df_prod["ANIO"].isin([2025, 2026])
].pivot_table(
    index="ANIO", columns="SEGMENTO", values="UNI_VEH_PRODUCCION",
    aggfunc="sum", fill_value=0
)

# Ritmo anualizado 2026 (10 meses -> 12 meses) para continuidad visual honesta
suv_26_pace  = piv_ytd.loc[2026, "SUV's"] * (12 / 10)
comp_26_pace = piv_ytd.loc[2026, "Compactos"] * (12 / 10)

segmentos_4 = ["SUV's", "Compactos", "Pick Ups", "Subcompactos"]

# =========================================================================
# VERSIÓN 1 · DEFAULT: lo que Bokeh entrega sin intervención (mostrar datos)
# =========================================================================
p1 = figure(width=760, height=380, title="Producción de vehículos ligeros por segmento (2005–2026)")
for i, seg in enumerate(["Compactos", "Minivans", "Pick Ups", "SUV's", "Subcompactos"]):
    p1.line(piv_all["ANIO"], piv_all[seg], line_width=2, color=Category10[5][i], legend_label=seg)
p1.legend.location = "top_left"
show(p1)

# =========================================================================
# VERSIÓN 2 · DECLUTTER + CONSISTENCIA TEMPORAL (Años completos 2005–2025)
# =========================================================================
p2 = figure(width=760, height=380, y_range=(0, 2_100_000), x_range=(2004.5, 2025.5),
            title="Producción anual de vehículos ligeros en México (2005–2025)")
for i, seg in enumerate(segmentos_4):
    p2.line(piv_25["ANIO"], piv_25[seg], line_width=2.5, color=Category10[4][i], legend_label=seg)
declutter(p2)
p2.yaxis.formatter = NumeralTickFormatter(format="0.0a")
p2.yaxis.axis_label = "Vehículos ensamblados"
p2.xaxis.ticker = [2005, 2010, 2015, 2020, 2025]
p2.legend.border_line_color = None
p2.legend.location = "top_left"
show(p2)

# =========================================================================
# VERSIÓN 3 · BASE NEUTRA EN GRIS + ETIQUETAS DIRECTAS (Adiós leyenda)
# =========================================================================
src_25 = ColumnDataSource(piv_25)
p3 = figure(width=780, height=380, y_range=(0, 2_100_000), x_range=(2004.5, 2029.8))
for seg in segmentos_4:
    p3.line("ANIO", seg, source=src_25, line_width=2.5, color=GREY)
    y_fin = piv_25[seg].iloc[-1]
    y_off = 8 if seg == "Pick Ups" else (-10 if seg == "Compactos" else 0)
    p3.add_layout(Label(
        x=2025, y=y_fin, text=f"{seg} ({y_fin/1e6:.2f}M)",
        x_offset=8, y_offset=y_off, text_baseline="middle",
        text_font_size="11px", text_color=TEXT_GREY
    ))
declutter(p3)
p3.yaxis.formatter = NumeralTickFormatter(format="0.0a")
p3.yaxis.axis_label = "Vehículos ensamblados"
p3.xaxis.ticker = [2005, 2010, 2015, 2020, 2025]
action_title(p3, "Evolución de la manufactura automotriz por segmento (2005–2025)",
             "Unidades ensambladas en México · Años completos")
show(p3)

# =========================================================================
# VERSIÓN 4 · FOCUS 1: EL VIEJO RÉGIMEN (Dominio de Compactos 2005–2016)
# =========================================================================
p4 = figure(width=780, height=390, y_range=(0, 2_100_000), x_range=(2004.5, 2029.8))
for seg in segmentos_4:
    es_comp = (seg == "Compactos")
    p4.line("ANIO", seg, source=src_25,
            line_width=3.5 if es_comp else 1.8,
            color=DARK if es_comp else LIGHT_GREY)
    y_fin = piv_25[seg].iloc[-1]
    y_off = 8 if seg == "Pick Ups" else (-10 if seg == "Compactos" else 0)
    p4.add_layout(Label(
        x=2025, y=y_fin, text=f"{seg} ({y_fin/1e6:.2f}M)",
        x_offset=8, y_offset=y_off, text_baseline="middle",
        text_font_size="11px",
        text_color=DARK if es_comp else TEXT_GREY,
        text_font_style="bold" if es_comp else "normal"
    ))

p4.scatter([2005, 2016], [piv_25.loc[piv_25["ANIO"]==2005, "Compactos"].item(),
                         piv_25.loc[piv_25["ANIO"]==2016, "Compactos"].item()],
           size=8, color=DARK)
p4.add_layout(Label(
    x=2016, y=1_194_467, text="Pico Compactos (2016): 1.19M (46% del total)",
    x_offset=-120, y_offset=14, text_font_size="11px", text_color=DARK, text_font_style="bold"
))
declutter(p4)
p4.yaxis.formatter = NumeralTickFormatter(format="0.0a")
p4.yaxis.axis_label = "Vehículos ensamblados"
p4.xaxis.ticker = [2005, 2010, 2015, 2020, 2025]
action_title(p4, "Durante más de una década (2005–2016), los compactos lideraron el ensamble",
             "En 2005 representaban 61.6% de la producción, frente a apenas 5.5% de las SUV's")
show(p4)

# =========================================================================
# VERSIÓN 5 · REFOCUS 2: EL GRAN PIVOTE HACIA LAS SUV's (Cruce de 2018)
# =========================================================================
p5 = figure(width=780, height=390, y_range=(0, 2_100_000), x_range=(2004.5, 2029.8))
for seg in segmentos_4:
    es_suv  = (seg == "SUV's")
    es_comp = (seg == "Compactos")
    color = BLUE if es_suv else (GREY if es_comp else LIGHT_GREY)
    width = 3.8 if es_suv else (2.0 if es_comp else 1.6)
    p5.line("ANIO", seg, source=src_25, line_width=width, color=color)
    y_fin = piv_25[seg].iloc[-1]
    y_off = 8 if seg == "Pick Ups" else (-10 if seg == "Compactos" else 0)
    p5.add_layout(Label(
        x=2025, y=y_fin, text=f"{seg} ({y_fin/1e6:.2f}M)",
        x_offset=8, y_offset=y_off, text_baseline="middle",
        text_font_size="11px",
        text_color=BLUE if es_suv else TEXT_GREY,
        text_font_style="bold" if es_suv else "normal"
    ))

p5.scatter([2005, 2018, 2025],
           [piv_25.loc[piv_25["ANIO"]==2005, "SUV's"].item(),
            piv_25.loc[piv_25["ANIO"]==2018, "SUV's"].item(),
            piv_25.loc[piv_25["ANIO"]==2025, "SUV's"].item()],
           size=8, color=BLUE)
p5.add_layout(Label(
    x=2018, y=1_454_356, text="2018: Cruce histórico (1.45M SUV's > 1.10M Compactos)",
    x_offset=-215, y_offset=14, text_font_size="11px", text_color=BLUE, text_font_style="bold"
))
declutter(p5)
p5.yaxis.formatter = NumeralTickFormatter(format="0.0a")
p5.yaxis.axis_label = "Vehículos ensamblados"
p5.xaxis.ticker = [2005, 2010, 2015, 2020, 2025]
action_title(p5, "Desde 2018 las SUV's desplazaron a los compactos y crecieron 27x",
             "Pasaron de 67 mil unidades en 2005 a 1.82 millones en 2025 (57.6% del total)")
show(p5)

# =========================================================================
# VERSIÓN 6 · CONTEXTO Y CHOQUES ESTRUCTURALES (Pandemia 2020 + Chips 2021–22)
# =========================================================================
p6 = figure(width=780, height=400, y_range=(0, 2_150_000), x_range=(2004.5, 2029.8))
p6.add_layout(BoxAnnotation(left=2019.7, right=2022.3, fill_color=LIGHT_GREY, fill_alpha=0.35, line_color=None))

for seg in segmentos_4:
    es_suv  = (seg == "SUV's")
    es_comp = (seg == "Compactos")
    color = BLUE if es_suv else (GREY if es_comp else LIGHT_GREY)
    width = 3.8 if es_suv else (2.0 if es_comp else 1.6)
    p6.line("ANIO", seg, source=src_25, line_width=width, color=color)
    y_fin = piv_25[seg].iloc[-1]
    y_off = 8 if seg == "Pick Ups" else (-10 if seg == "Compactos" else 0)
    p6.add_layout(Label(
        x=2025, y=y_fin, text=f"{seg} ({y_fin/1e6:.2f}M)",
        x_offset=8, y_offset=y_off, text_baseline="middle",
        text_font_size="11px",
        text_color=BLUE if es_suv else TEXT_GREY,
        text_font_style="bold" if es_suv else "normal"
    ))

p6.add_layout(Label(
    x=2021, y=2_050_000, text="COVID-19 (2020) y\nCrisis de Chips (2021–22)",
    text_align="center", text_baseline="top", text_font_size="10.5px",
    text_color=DARK, text_font_style="bold"
))
p6.add_layout(Label(
    x=2022, y=420_636, text="Armadoras priorizan chips en SUV's;\nCompactos cae a mínimo (421 mil; -65%)",
    x_offset=-210, y_offset=-36, text_font_size="10.5px", text_color=DARK
))
declutter(p6)
p6.yaxis.formatter = NumeralTickFormatter(format="0.0a")
p6.yaxis.axis_label = "Vehículos ensamblados"
p6.xaxis.ticker = [2005, 2010, 2015, 2020, 2025]
action_title(p6, "La escasez de semiconductores (2021–2022) aceleró el abandono de compactos",
             "Los fabricantes asignaron microchips escasos a SUV's de mayor margen de exportación")
show(p6)

# =========================================================================
# VERSIÓN 7 · HISTORIA COMPLETA + ALERTA ARANCELARIA 2026 Y HOVER INTERACTIVO
# =========================================================================
p7 = figure(width=840, height=430, y_range=(0, 2_200_000), x_range=(2004.5, 2031.2))
p7.add_layout(BoxAnnotation(left=2019.7, right=2022.3, fill_color=LIGHT_GREY, fill_alpha=0.30, line_color=None))

for seg in segmentos_4:
    es_suv  = (seg == "SUV's")
    es_comp = (seg == "Compactos")
    color = BLUE if es_suv else (GREY if es_comp else LIGHT_GREY)
    width = 3.8 if es_suv else (2.0 if es_comp else 1.6)
    r = p7.line("ANIO", seg, source=src_25, line_width=width, color=color, name=seg)

# Segmento 2025 -> 2026 (ritmo anualizado Ene-Oct) en naranja de alerta
suv_25_val  = piv_25.loc[piv_25["ANIO"]==2025, "SUV's"].item()
comp_25_val = piv_25.loc[piv_25["ANIO"]==2025, "Compactos"].item()

p7.line([2025, 2026], [suv_25_val, suv_26_pace], line_width=3.2, line_dash="dashed", color=ORANGE)
p7.scatter([2026], [suv_26_pace], size=9, color=ORANGE)

p7.line([2025, 2026], [comp_25_val, comp_26_pace], line_width=2.0, line_dash="dashed", color=ORANGE)
p7.scatter([2026], [comp_26_pace], size=7, color=ORANGE)

# Línea vertical de evento: Aranceles Enero 2026
p7.add_layout(Span(location=2025.85, dimension="height", line_color=ORANGE, line_dash="dotted", line_width=1.5))
p7.add_layout(Label(
    x=2025.85, y=2_130_000, text="Aranceles Ene 2026:\nSUV's -7.4% YTD\nCompactos -16.1% YTD",
    x_offset=-125, text_baseline="top", text_font_size="10.5px",
    text_color=ORANGE, text_font_style="bold"
))
p7.add_layout(Label(
    x=2021, y=2_130_000, text="COVID + Chips\n(2020–2022)",
    text_align="center", text_baseline="top", text_font_size="10px", text_color=TEXT_GREY
))

# Etiquetas directas finales
p7.add_layout(Label(x=2026, y=suv_26_pace, text="SUV's (58% mix · 93.5% exportada)",
                    x_offset=10, text_baseline="middle", text_font_size="11px",
                    text_color=ORANGE, text_font_style="bold"))
p7.add_layout(Label(x=2026, y=comp_26_pace, text="Compactos (-16.1% YTD)",
                    x_offset=10, y_offset=6, text_baseline="middle", text_font_size="10.5px", text_color=TEXT_GREY))
p7.add_layout(Label(x=2025, y=piv_25.loc[piv_25["ANIO"]==2025, "Pick Ups"].item(),
                    text="Pick Ups (+3.1% YTD)", x_offset=48, y_offset=-8,
                    text_baseline="middle", text_font_size="10.5px", text_color=TEXT_GREY))
p7.add_layout(Label(x=2025, y=piv_25.loc[piv_25["ANIO"]==2025, "Subcompactos"].item(),
                    text="Subcompactos", x_offset=48,
                    text_baseline="middle", text_font_size="10.5px", text_color=TEXT_GREY))

# Nota metodológica transparente en pantalla (Honesty about coverage)
p7.add_layout(Title(
    text="Fuente: INEGI RAIAVL (2005–Oct 2026). Nota: 2026 proyectado a ritmo Ene–Oct (10 meses; incluye cifras preliminares/revisadas).",
    text_font_size="10px", text_color=TEXT_GREY, text_font_style="italic"
), "below")

declutter(p7)
p7.yaxis.formatter = NumeralTickFormatter(format="0.0a")
p7.yaxis.axis_label = "Vehículos ensamblados"
p7.xaxis.ticker = [2005, 2010, 2015, 2020, 2026]
action_title(
    p7,
    "El pivote a SUV's elevó el margen, pero concentró el riesgo arancelario en 2026",
    "Recomendación: diversificar destinos fuera de EE. UU. (hoy 79.9% de exportación) y proteger plataformas híbridas"
)

# Hover interactivo aun con la barra oculta
p7.add_tools(HoverTool(tooltips=[("Año", "@ANIO"), ("SUV's", "@{SUV's}{0,0}"), ("Compactos", "@Compactos{0,0}"), ("Pick Ups", "@{Pick Ups}{0,0}")]))
show(p7)

# Exportar la versión final como HTML interactivo independiente
output_file("storytelling_automotriz_mexico_2005_2026.html", title="Storytelling Industria Automotriz México")
save(p7)

Loading BokehJS ...

'c:\\Users\\Rafael Martinez\\Documents\\Concentracion_AnDatos_IA\\Data Visualization\\storytelling_automotriz_mexico_2005_2026.html'